# **Removing Duplicate/Inadequate Rows**

In [ ]:
import pandas as pd

df = pd.read_csv("scraped_jobs.csv")
df = df.dropna(subset=["description"])

initial_rows = len(df)
print("Initial dataset size:",initial_rows)

df = df.drop_duplicates(
    subset=["role", "description"],
    keep="first"
)



import re

def has_valid_words(text, min_words=10):
    words = re.findall(r"[a-zA-Z]{2,}", text)
    return len(words) >= min_words

df = df[df["description"].apply(has_valid_words)]

len(df)



df["desc_length"] = df["description"].str.len()
avg_length = df["desc_length"].mean()
MIN_LENGTH = avg_length * 0.4
df = df[df["desc_length"] >= MIN_LENGTH]

def valid_job_title(title):
    if pd.isna(title):
        return False
    title = title.lower()
    if len(title) < 3:
        return False
    if not re.search(r"[a-zA-Z]", title):
        return False
    return True

df = df[df["role"].apply(valid_job_title)]

print(f"Final dataset size: {len(df)} job postings")



Initial Dataset Size: 954
Final dataset size: 764 job postings


# **Tokenization & Lemmatization**

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)      # remove html
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc 
            if not token.is_stop and token.is_alpha]

df["tokens"] = df["clean_description"].apply(lemmatize)


# **Role Normalization**


In [ ]:
import re

def clean_job_title(title):
    title = title.lower()
    title = re.sub(r"@.*", "", title)                # remove company
    title = re.sub(r"\(.*?\)", "", title)            # remove brackets
    title = re.sub(r"[-/|]", " ", title)              # separators
    title = re.sub(r"\b(senior|jr|junior|lead|tech lead|principal)\b", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

df["clean_title"] = df["role"].apply(clean_job_title)

ROLE_KEYWORDS = {
    # --- Engineering & Development ---
    "software engineer": ["software engineer", "software developer", "swe", "developer", "application developer", "systems programmer"],
    "web developer": ["web developer", "frontend developer", "backend developer", "full stack developer", "ui developer", "javascript developer"],
    "mobile developer": ["mobile developer", "android developer", "ios developer", "react native developer", "flutter developer", "swift developer"],
    "game developer": ["game developer", "unity developer", "unreal engine developer", "game programmer", "graphics engineer"],
    "embedded engineer": ["embedded engineer", "firmware engineer", "iot developer", "hardware engineer", "systems engineer"],

    # --- Data & AI ---
    "data scientist": ["data scientist", "machine learning engineer", "ml engineer", "ai engineer", "nlp engineer", "computer vision engineer", "deep learning engineer"],
    "data analyst": ["data analyst", "business analyst", "bi analyst", "product analyst", "data visualizer", "tableau developer"],
    "data engineer": ["data engineer", "etl developer", "big data engineer", "data architect", "analytics engineer"],
    "database administrator": ["dba", "database administrator", "database engineer", "sql developer"],

    # --- Infrastructure, Cloud & DevOps ---
    "devops engineer": ["devops engineer", "site reliability engineer", "sre", "platform engineer", "automation engineer", "build engineer"],
    "cloud engineer": ["cloud engineer", "cloud architect", "aws architect", "azure engineer", "gcp architect", "cloud consultant"],
    "network engineer": ["network engineer", "network architect", "systems administrator", "sysadmin", "infrastructure engineer"],

    # --- Security ---
    "cybersecurity engineer": ["security engineer", "cybersecurity", "information security", "soc analyst", "pentester", "ethical hacker", "application security engineer", "security architect"],
    "compliance officer": ["it compliance", "grc analyst", "it auditor", "privacy engineer"],

    # --- Quality & Testing ---
    "qa engineer": ["qa engineer", "quality assurance", "software tester", "automation tester", "sdet", "manual tester", "performance engineer"],

    # --- Product & Management ---
    "product manager": ["product manager", "pm", "technical product manager", "product owner"],
    "project manager": ["project manager", "it project manager", "scrum master", "agile coach", "delivery manager"],
    "it manager": ["it manager", "cto", "cio", "engineering manager", "vpe", "it director"],

    # --- Design & UX ---
    "ux designer": ["ux designer", "ui designer", "product designer", "user researcher", "interaction designer", "ux writer"],

    # --- Specialized Platforms & ERP ---
    "salesforce developer": ["salesforce developer", "sfdc developer", "salesforce admin", "crm developer"],
    "erp consultant": ["sap consultant", "oracle functional consultant", "dynamics 365 developer", "erp analyst"],
    "service management": ["itsm manager", "servicenow developer", "itil consultant"]
}

def normalize_role(title):
    for canonical_role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            if kw in title:
                return canonical_role
    return "other"

df["normalized_role"] = df["clean_title"].apply(normalize_role)